# TEC206 Intermediate Programming — Week 09
## Unit Testing, Automated Testing & Code Quality

**Lecture format:** explanation → demonstration → flowchart → guided practice → challenge → self-check quiz  
**Language:** Python 3  
**Prerequisite:** variables, data types, conditionals, loops, functions, classes, and exception handling

---

### Why this workbook exists

Writing code is only half of software development. We also need evidence that the code behaves correctly.

When programs are small, we often test them manually:

1. run the program,
2. type an input,
3. look at the output,
4. decide whether it seems correct,
5. repeat with another input.

That works at first, but it becomes slow and unreliable as a program grows.

This week introduces **automated testing** and especially **unit testing**: writing code that tests other code. We begin with Python's built-in `assert` statement, move to the standard-library `unittest` framework, then use modern tools such as `pytest`, Hypothesis, coverage tools, linters, type checkers, security scanners, complexity analyzers, mutation testing, and automated unit-test generation.

By the end, you should be able to build a small automated quality pipeline rather than checking every result by hand.


> ## How to use this notebook
>
> 1. Run cells from top to bottom.
> 2. **Predict the result before running each demonstration.**
> 3. For every test, identify the **input**, **expected result**, and **actual result**.
> 4. When a test fails, treat the failure as useful information.
> 5. Change values and deliberately create failures.
> 6. Complete each activity before reading the suggested solution.
> 7. The final quiz uses `ipywidgets` when available; a text fallback is also included.
>
> ### Important principle
>
> **A program that runs without crashing is not necessarily correct.**
>
> Syntax checkers, linters, formatters, and type checkers are helpful, but they cannot generally prove that your business logic is correct. Tests compare behaviour against requirements.


# 1. Learning Outcomes

By the end of Week 09, you should be able to:

- explain why software testing is necessary;
- distinguish **manual testing** from **automated testing**;
- explain what a **test case**, **test suite**, **test runner**, and **unit** are;
- identify the **Arrange → Act → Assert** structure of a test;
- use Python's `assert` statement for simple checks;
- explain what `AssertionError` means;
- test normal inputs, boundary values, zero, negative values, and exceptions;
- build a test class with `unittest.TestCase`;
- use `assertEqual`, `assertTrue`, `assertFalse`, `assertAlmostEqual`, and `assertRaises`;
- run `unittest` safely inside a Jupyter notebook;
- write modern tests using `pytest`;
- use `pytest.mark.parametrize` to test many input/output pairs;
- explain fixtures and test isolation;
- use **property-based testing** with Hypothesis;
- explain **code coverage** and why 100% coverage does not guarantee correctness;
- distinguish unit testing from linting, formatting, type checking, security scanning, and complexity analysis;
- use tools such as Ruff, mypy, Bandit, and Radon;
- explain **mutation testing** with `mutmut`;
- explain automated test generation with **Pynguin** and its safety limitations;
- describe how tests can run automatically in Continuous Integration (CI).


# 2. What Is Software Testing?

**Software testing** is the process of checking whether software behaves as required.

A test normally asks:

> **For this input and situation, did the program produce the expected behaviour?**

Examples:

| Requirement | Example input | Expected behaviour |
|---|---:|---|
| Multiply a number by 10 | `10` | `100` |
| Multiply a number by 10 | `0` | `0` |
| Multiply a number by 10 | `-10` | `-100` |
| Add two numbers | `5, 3` | `8` |
| Divide by zero | `10, 0` | Raise `ZeroDivisionError` |
| Check age eligibility | `17` | `False` |
| Check age eligibility | `18` | `True` |

Testing is not only about finding crashes. A program can execute perfectly and still calculate the wrong answer.


## 2.1 Three Important Categories of Problems

| Problem | Example | Can testing help? |
|---|---|---|
| **Syntax error** | Missing `:` | Usually found before normal execution |
| **Runtime error** | `10 / 0` | Yes — tests can check expected exceptions |
| **Logic error** | Discount adds 20% instead of subtracting 20% | **Yes — this is a major purpose of tests** |

Consider:

```python
def apply_discount(price):
    return price * 1.20
```

The code is valid Python. A formatter may format it. A linter may accept it. A type checker may accept it.

But if the requirement is “apply a 20% discount”, the correct result for `$100` is `$80`, not `$120`.

A behavioural test can expose this error immediately.


In [ ]:
def apply_discount_buggy(price):
    # Intentionally incorrect: adds 20% instead of removing 20%.
    return price * 1.20

actual = apply_discount_buggy(100)
expected = 80

print("Expected:", expected)
print("Actual  :", actual)
print("Correct? :", actual == expected)


# 3. Manual Testing vs Automated Testing

## Manual testing

A person runs the program, enters values, observes output, and decides whether it is correct.

## Automated testing

We write another piece of code that:

1. calls the program/function,
2. supplies test data,
3. compares the actual result with the expected result,
4. reports **PASS** or **FAIL** automatically.

| Feature | Manual testing | Automated testing |
|---|---|---|
| Human repeatedly enters inputs | Yes | Usually no |
| Fast to repeat hundreds of times | No | Yes |
| Good for exploratory/user-experience checks | Yes | Sometimes |
| Consistent every run | Depends on person | Yes |
| Useful after every code change | Time-consuming | Excellent |
| Can run in CI | No | Yes |
| Initial setup effort | Low | Higher |
| Long-term repeatability | Low | High |

### Key idea

Automation does **not** remove all manual testing. It automates checks that are repetitive and precisely defined.


## 3.1 Testing Process Flowchart

```text
MANUAL TESTING
──────────────
Run program
    │
    ▼
Enter input manually
    │
    ▼
Observe output
    │
    ▼
Compare with expectation
    │
    ▼
Repeat for another case


AUTOMATED TESTING
─────────────────
Write test once
    │
    ▼
Test runner executes test cases
    │
    ▼
Actual result compared with expected result
    │
    ├──────────────┐
    ▼              ▼
  PASS           FAIL
                   │
                   ▼
            Investigate / fix
                   │
                   ▼
             Run tests again
```


In [ ]:
# Visual comparison of manual and automated testing.

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 5))
ax.axis("off")

manual = [
    ("Run program", 0.18, 0.82),
    ("Enter input", 0.18, 0.62),
    ("Observe output", 0.18, 0.42),
    ("Compare manually", 0.18, 0.22),
]
automated = [
    ("Run test suite", 0.72, 0.82),
    ("Call code automatically", 0.72, 0.62),
    ("Compare expected vs actual", 0.72, 0.42),
    ("PASS / FAIL report", 0.72, 0.22),
]

for label, x, y in manual + automated:
    ax.text(
        x, y, label, ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.45")
    )

for group in (manual, automated):
    for (_, x1, y1), (_, x2, y2) in zip(group, group[1:]):
        ax.annotate(
            "", xy=(x2, y2 + 0.06), xytext=(x1, y1 - 0.06),
            arrowprops=dict(arrowstyle="->")
        )

ax.text(0.18, 0.95, "MANUAL", ha="center", weight="bold", fontsize=13)
ax.text(0.72, 0.95, "AUTOMATED", ha="center", weight="bold", fontsize=13)
plt.show()


# 4. What Is a Unit?

A **unit** is a small piece of software that can be tested independently.

In an introductory Python program, a unit is often:

- one function;
- one method;
- one small class.

### Door analogy

Imagine testing a door system.

The full system includes:

- the **door panel**;
- **hinges**;
- a **handle**;
- a **lock**;
- the frame.

If the door does not close, testing the entire doorway only tells us that “something is wrong”.

Unit testing asks smaller questions:

- Do the hinges rotate?
- Does the handle retract the latch?
- Does the lock change between locked and unlocked?
- Does the door report its state correctly?

Testing smaller parts makes failures easier to locate.


## 4.1 Unit vs Integration vs End-to-End Testing

```text
                 /\
                /  \
               / E2E\        Full user workflow
              /------\
             /Integration\   Components working together
            /------------\
           /  Unit Tests   \ Small functions/classes
          /________________\
```

For this week, our main focus is the broad base: **unit tests**.

- **Unit test:** Is one function/class correct?
- **Integration test:** Do multiple components work together?
- **End-to-end test:** Does the complete application workflow work from the user's perspective?

A healthy project may use all three.


# 5. Anatomy of a Test: Arrange → Act → Assert

A useful pattern is **AAA**:

1. **Arrange** — prepare the input and expected result.
2. **Act** — call the function.
3. **Assert** — compare actual behaviour with expected behaviour.

Example:

```python
# Arrange
number = 20
expected = 200

# Act
actual = multiply_by_10(number)

# Assert
assert actual == expected
```


In [ ]:
def multiply_by_10(number):
    return number * 10

# Arrange
number = 20
expected = 200

# Act
actual = multiply_by_10(number)

# Assert
assert actual == expected

print("Test passed:", actual, "==", expected)


# 6. Python's `assert` Statement

The simplest automated check in Python is:

```python
assert condition
```

- If the condition is `True`, execution continues.
- If the condition is `False`, Python raises an **`AssertionError`**.

We can also include a message:

```python
assert condition, "Helpful failure message"
```

`assert` is useful for learning and for checks inside tests.


In [ ]:
def multiply_by_10(number):
    return number * 10

assert multiply_by_10(10) == 100
assert multiply_by_10(0) == 0
assert multiply_by_10(-10) == -100

print("All three assert checks passed.")


## 6.1 What Does a Failed Assertion Look Like?

The next cell intentionally checks the wrong expectation, but catches the error so the notebook can continue.


In [ ]:
try:
    assert multiply_by_10(10) == 200, "10 multiplied by 10 should be 100, not 200."
except AssertionError as exc:
    print("A test failed.")
    print("Exception type:", type(exc).__name__)
    print("Message       :", exc)


## 6.2 Assertions Are Not General Input Validation

Do **not** use `assert` as the main mechanism for:

- validating untrusted user input;
- enforcing security rules;
- checking permissions;
- enforcing business rules that must always execute.

Python can be run with optimization (`python -O`), which can remove ordinary `assert` statements.

For required validation, use normal control flow and raise an appropriate exception.

```python
def set_age(age):
    if age < 0:
        raise ValueError("Age cannot be negative")
```

Use assertions mainly for tests and developer invariants.


# 7. Designing Good Test Cases

One successful example is rarely enough.

For a numeric function, consider:

- normal positive values;
- zero;
- negative values;
- boundaries;
- very large/small values where relevant;
- wrong types if the function promises to reject them;
- expected exceptions.

We often talk about:

- **happy path** — normal expected use;
- **edge case** — unusual but valid input;
- **invalid case** — input that should be rejected.


In [ ]:
def subtract(a, b):
    return a - b

test_cases = [
    (10, 3, 7),      # normal positive
    (3, 10, -7),     # negative result
    (5, 5, 0),       # boundary: equal values
    (0, 0, 0),       # zero
    (-5, -3, -2),    # negative inputs
]

for a, b, expected in test_cases:
    actual = subtract(a, b)
    assert actual == expected
    print(f"PASS: subtract({a}, {b}) -> {actual}")


## 7.1 Activity — Predict Before Running

What should each expression return?

1. `multiply_by_10(7)`
2. `multiply_by_10(0)`
3. `multiply_by_10(-4)`
4. `subtract(8, 3)`
5. `subtract(3, 8)`

Now add assertions for all five cases.


In [ ]:
# YOUR TURN
# Add five assertions here.

# assert ...


### Suggested solution


In [ ]:
assert multiply_by_10(7) == 70
assert multiply_by_10(0) == 0
assert multiply_by_10(-4) == -40
assert subtract(8, 3) == 5
assert subtract(3, 8) == -5

print("Suggested-solution tests passed.")


# 8. Why Use a Testing Framework?

Plain `assert` statements are useful, but a framework gives us more structure.

A testing framework can provide:

- automatic test discovery;
- groups of tests;
- setup and cleanup;
- clear failure messages;
- checks for expected exceptions;
- test reports;
- selective test execution;
- integration with IDEs and CI systems.

Python includes **`unittest`** in the standard library, so no `pip install` is required.


# 9. Unit Testing with Python's `unittest`

A typical `unittest` test:

1. imports `unittest`;
2. defines a class that inherits from `unittest.TestCase`;
3. defines methods whose names normally begin with `test`;
4. uses assertion methods such as `assertEqual`;
5. lets a test runner execute the suite.

Typical assertion methods:

| Method | Meaning |
|---|---|
| `self.assertEqual(a, b)` | `a == b` |
| `self.assertNotEqual(a, b)` | `a != b` |
| `self.assertTrue(x)` | `x` is truthy |
| `self.assertFalse(x)` | `x` is falsy |
| `self.assertIsNone(x)` | `x is None` |
| `self.assertIn(a, b)` | `a in b` |
| `self.assertAlmostEqual(a, b)` | approximately equal floats |
| `self.assertRaises(Error)` | expected exception is raised |


In [ ]:
import unittest

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply_by_10(number):
    return number * 10

def divide(a, b):
    return a / b


class TestMathOperations(unittest.TestCase):

    def test_addition(self):
        self.assertEqual(add(2, 3), 5)

    def test_subtraction(self):
        self.assertEqual(subtract(10, 4), 6)

    def test_multiply_by_10_positive(self):
        self.assertEqual(multiply_by_10(10), 100)

    def test_multiply_by_10_zero(self):
        self.assertEqual(multiply_by_10(0), 0)

    def test_multiply_by_10_negative(self):
        self.assertEqual(multiply_by_10(-10), -100)

    def test_division(self):
        self.assertEqual(divide(10, 2), 5)

    def test_divide_by_zero(self):
        with self.assertRaises(ZeroDivisionError):
            divide(10, 0)


# Jupyter-friendly runner.
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestMathOperations)
result = unittest.TextTestRunner(verbosity=2).run(suite)

print("\nSuccessful?", result.wasSuccessful())


## 9.1 Why Are Test Methods Named `test_...`?

Test discovery tools use naming conventions.

For `unittest`, methods beginning with `test` are automatically recognised as test methods by the normal loader.

```python
def test_addition(self):
    ...
```

is discovered, while a helper method such as:

```python
def calculate_expected_value(self):
    ...
```

is not automatically treated as a test.


## 9.2 `setUp()` — Preparing Fresh State for Each Test

Sometimes several tests need the same object.

`setUp()` runs before **each** test method. This helps keep tests independent.


In [ ]:
class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self.balance += amount

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        self.balance -= amount


class TestBankAccount(unittest.TestCase):

    def setUp(self):
        self.account = BankAccount(100)

    def test_initial_balance(self):
        self.assertEqual(self.account.balance, 100)

    def test_deposit(self):
        self.account.deposit(50)
        self.assertEqual(self.account.balance, 150)

    def test_withdraw(self):
        self.account.withdraw(40)
        self.assertEqual(self.account.balance, 60)

    def test_invalid_deposit(self):
        with self.assertRaises(ValueError):
            self.account.deposit(-10)

    def test_overdraw(self):
        with self.assertRaises(ValueError):
            self.account.withdraw(1000)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestBankAccount)
unittest.TextTestRunner(verbosity=2).run(suite)


# 10. A Test Failure Is Information

Suppose we accidentally write:

```python
def is_even(number):
    return number % 2 == 1
```

The program runs. There is no syntax error. But the logic is reversed.

A unit test should expose it.


In [ ]:
def is_even_buggy(number):
    return number % 2 == 1   # intentionally wrong

class TestBuggyEvenFunction(unittest.TestCase):
    def test_even_number(self):
        self.assertTrue(is_even_buggy(4))

    def test_odd_number(self):
        self.assertFalse(is_even_buggy(5))

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestBuggyEvenFunction)
result = unittest.TextTestRunner(verbosity=2).run(suite)

print("\nThe failures above are expected for this demonstration.")
print("They show that valid Python can still contain incorrect logic.")


In [ ]:
def is_even(number):
    return number % 2 == 0

assert is_even(4) is True
assert is_even(5) is False
assert is_even(0) is True
assert is_even(-2) is True

print("Fixed implementation passes the checks.")


# 11. Modern Python Testing with `pytest`

`pytest` is a widely used third-party Python testing framework.

Why students often find it convenient:

- tests can be ordinary functions;
- ordinary Python `assert` statements are used;
- test discovery is automatic;
- assertion failures are displayed with detailed context;
- fixtures make setup reusable;
- parametrization makes repeated cases concise;
- it can run many existing `unittest` test suites.

Install:

```bash
python -m pip install pytest
```

A minimal pytest test:

```python
def test_addition():
    assert add(2, 3) == 5
```


## 11.1 Optional Lab Installation

Run this only if these packages are not already installed.

The standard-library `unittest` module does **not** need installation.


In [ ]:
# Uncomment this line if you want to install the recommended lab tools.
# %pip install -q pytest pytest-cov hypothesis coverage ruff mypy bandit radon nbformat


## 11.2 A Real `pytest` Mini-Project

The next cell writes two small Python files into a local teaching folder:

```text
week09_testing_demo/
├── math_utils.py
└── test_math_utils.py
```

This mirrors how testing is normally organised outside a notebook.


In [ ]:
from pathlib import Path

demo_dir = Path("week09_testing_demo")
demo_dir.mkdir(exist_ok=True)

(demo_dir / "math_utils.py").write_text(
"""def multiply_by_10(number):
    return number * 10

def add(a, b):
    return a + b

def divide(a, b):
    return a / b
""",
encoding="utf-8",
)

(demo_dir / "test_math_utils.py").write_text(
"""import pytest
from math_utils import multiply_by_10, add, divide

def test_multiply_by_10():
    assert multiply_by_10(10) == 100

def test_zero():
    assert multiply_by_10(0) == 0

def test_negative():
    assert multiply_by_10(-10) == -100

@pytest.mark.parametrize(
    "a,b,expected",
    [
        (1, 2, 3),
        (0, 0, 0),
        (-2, 5, 3),
        (100, -25, 75),
    ],
)
def test_add_many_cases(a, b, expected):
    assert add(a, b) == expected

def test_divide_by_zero():
    with pytest.raises(ZeroDivisionError):
        divide(10, 0)
""",
encoding="utf-8",
)

print("Created:")
for path in sorted(demo_dir.iterdir()):
    print(" -", path)


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pytest") is None:
    print("pytest is not installed.")
    print("Install with: python -m pip install pytest")
else:
    completed = subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd="week09_testing_demo",
        text=True,
        capture_output=True,
    )
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)


# 12. `pytest.mark.parametrize` — Many Cases, One Test

Instead of copying the same test body many times, we can express cases as data:

```python
@pytest.mark.parametrize(
    "a,b,expected",
    [
        (1, 2, 3),
        (0, 0, 0),
        (-2, 5, 3),
    ],
)
def test_add(a, b, expected):
    assert add(a, b) == expected
```

Each row becomes a separate test case.


# 13. Fixtures — Reusable Test Setup

A **fixture** provides prepared data or objects to a test.

```python
@pytest.fixture
def account():
    return BankAccount(100)

def test_deposit(account):
    account.deposit(50)
    assert account.balance == 150
```

Fixtures are useful for:

- temporary files;
- sample objects;
- database connections;
- mock services;
- reusable datasets.

The important design goal is **test isolation**: one test should not secretly depend on another test running first.


# 14. Property-Based Testing with Hypothesis

Traditional unit tests choose specific examples:

```python
assert add(2, 3) == 5
assert add(-1, 1) == 0
```

**Property-based testing** asks:

> What property should hold for *many automatically generated inputs*?

For addition, one property is **commutativity**:

```text
a + b == b + a
```

Hypothesis generates many values — including edge cases you may not think to write manually.

Install:

```bash
python -m pip install hypothesis
```


In [ ]:
import importlib.util

if importlib.util.find_spec("hypothesis") is None:
    print("Hypothesis is not installed.")
    print("Install with: python -m pip install hypothesis")
else:
    from hypothesis import given, strategies as st

    @given(st.integers(), st.integers())
    def test_addition_is_commutative(a, b):
        assert add(a, b) == add(b, a)

    test_addition_is_commutative()
    print("Hypothesis property test completed successfully.")


## 14.1 Example Property for `multiply_by_10`

For every integer `n`:

```text
multiply_by_10(n) == n * 10
```

Another property is:

```text
multiply_by_10(-n) == -multiply_by_10(n)
```

### Important distinction

Hypothesis generates **test inputs** from properties you define.  
It does not magically know your business requirement. You still need to define meaningful properties.


# 15. Code Coverage

**Code coverage** measures which parts of your code executed while tests ran.

Common questions:

- Which lines were executed?
- Which branches of an `if/else` were executed?
- Which lines were never reached by the tests?

Typical install:

```bash
python -m pip install coverage pytest-cov
```

Typical pytest command:

```bash
python -m pytest --cov=. --cov-branch --cov-report=term-missing
```

### Critical warning

**100% coverage does not mean 100% correctness.**

A test can execute every line but make weak or incorrect assertions.

Coverage tells us **where tests went**, not whether the tests were intelligent.


## 15.1 Coverage Example

Suppose:

```python
def shipping_cost(total):
    if total >= 50:
        return 0
    return 10
```

If we test only `shipping_cost(100)`, we execute the free-shipping branch but not the `$10` branch.

Better cases include:

- `49` → `10`
- `50` → `0`  ← boundary
- `51` → `0`


In [ ]:
def shipping_cost(total):
    if total >= 50:
        return 0
    return 10

assert shipping_cost(49) == 10
assert shipping_cost(50) == 0
assert shipping_cost(51) == 0

print("Boundary tests passed.")


# 16. Testing Is Not the Same as Code Quality Analysis

You mentioned tools that can “scan the code”, “review it”, “beautify it”, or report possible errors. They solve different problems.

| Tool/type | Main question |
|---|---|
| Unit tests | **Does the code behave as required?** |
| Linter | Does the source contain suspicious patterns/style problems? |
| Formatter | Is the code formatted consistently? |
| Type checker | Are annotated types used consistently? |
| Security scanner | Does the code contain known risky patterns? |
| Coverage | Which code did tests execute? |
| Complexity analyzer | Which functions are becoming difficult to understand/test? |
| Mutation tester | Would the tests notice small intentional defects? |
| Test generator | Can useful test cases be generated automatically? |

No single tool replaces the others.


# 17. Ruff — Linting and Formatting

**Ruff** can inspect Python code for many common issues and can also format code.

Install:

```bash
python -m pip install ruff
```

Useful commands:

```bash
ruff check .
ruff check --fix .
ruff format .
ruff format --check .
```

Examples of issues a linter may identify:

- unused imports;
- undefined names;
- unnecessary code patterns;
- style problems;
- some bug-prone constructs.

A linter may improve code quality, but it does not know that your 20% discount should produce `$80`.


In [ ]:
from pathlib import Path
import shutil
import subprocess

quality_demo = Path("week09_testing_demo") / "quality_demo.py"
quality_demo.parent.mkdir(exist_ok=True)
quality_demo.write_text(
"""import os
import math

def greet(name):
    unused_value = 123
    return f"Hello, {name}"

def apply_discount(price: float, rate: float) -> float:
    return price * (1 + rate)
""",
encoding="utf-8",
)

print(quality_demo.read_text())


In [ ]:
ruff = shutil.which("ruff")
if ruff is None:
    print("Ruff is not installed.")
    print("Install with: python -m pip install ruff")
else:
    completed = subprocess.run(
        [ruff, "check", "week09_testing_demo/quality_demo.py"],
        text=True,
        capture_output=True,
    )
    print(completed.stdout or "No Ruff findings.")


# 18. mypy — Static Type Checking

Python allows optional **type hints**:

```python
def add(a: int, b: int) -> int:
    return a + b
```

A static type checker such as **mypy** checks annotated code without executing it.

Install:

```bash
python -m pip install mypy
```

Run:

```bash
mypy program.py
```

Correct types do not guarantee correct logic.


In [ ]:
typed_demo = Path("week09_testing_demo") / "typed_demo.py"
typed_demo.write_text(
"""def double(number: int) -> int:
    return number * 2

result = double("hello")
""",
encoding="utf-8",
)

mypy = shutil.which("mypy")
if mypy is None:
    print("mypy is not installed.")
    print("Install with: python -m pip install mypy")
else:
    completed = subprocess.run(
        [mypy, str(typed_demo)],
        text=True,
        capture_output=True,
    )
    print(completed.stdout or completed.stderr)


# 19. Bandit — Security-Oriented Static Analysis

**Bandit** scans Python source for common security-risk patterns.

Install:

```bash
python -m pip install bandit
```

Run recursively:

```bash
bandit -r path/to/code
```

Bandit is useful for security review, but it is not a substitute for behavioural tests or a full security audit.


In [ ]:
security_demo = Path("week09_testing_demo") / "security_demo.py"
security_demo.write_text(
"""import subprocess

def run_user_command(command):
    return subprocess.call(command, shell=True)
""",
encoding="utf-8",
)

bandit = shutil.which("bandit")
if bandit is None:
    print("Bandit is not installed.")
    print("Install with: python -m pip install bandit")
else:
    completed = subprocess.run(
        [bandit, "-q", str(security_demo)],
        text=True,
        capture_output=True,
    )
    print(completed.stdout or completed.stderr or "No findings.")


# 20. Radon — Complexity and Maintainability Metrics

**Radon** measures properties of Python code such as:

- cyclomatic complexity;
- Maintainability Index;
- raw source metrics;
- Halstead metrics.

Install:

```bash
python -m pip install radon
```

Examples:

```bash
radon cc -s program.py
radon mi -s program.py
```

Radon can also analyse code cells in Jupyter notebooks:

```bash
radon cc --include-ipynb --ipynb-cells .
```

### Why complexity matters for testing

A function with many `if`, `elif`, loops, and logical branches has many possible paths. More paths usually mean more test cases are needed.


In [ ]:
complexity_demo = Path("week09_testing_demo") / "complexity_demo.py"
complexity_demo.write_text(
"""def classify(number):
    if number < 0:
        if number < -100:
            return "very negative"
        return "negative"
    elif number == 0:
        return "zero"
    elif number < 10:
        return "small positive"
    elif number < 100:
        return "medium positive"
    return "large positive"
""",
encoding="utf-8",
)

radon = shutil.which("radon")
if radon is None:
    print("Radon is not installed.")
    print("Install with: python -m pip install radon")
else:
    completed = subprocess.run(
        [radon, "cc", "-s", str(complexity_demo)],
        text=True,
        capture_output=True,
    )
    print(completed.stdout or completed.stderr)


# 21. What Can Automatically Find a Logic Error?

Suppose:

```python
def calculate_tax(amount):
    return amount * 1.50
```

If the requirement is a 15% tax, the correct multiplier should be `1.15`.

| Tool | Likely to detect the business-logic mistake? |
|---|---|
| Python syntax parser | No |
| Formatter | No |
| Ruff | Usually no |
| mypy | No, types are valid |
| Bandit | No, not a security pattern |
| Radon | No, code is simple |
| Unit test expecting `115` from `100` | **Yes** |
| Property/specification-based test | **Yes, if the property captures the requirement** |

Static tools can identify many mistakes, but there is no general scanner that can infer every intended requirement from source code alone.

That is why we write executable specifications in the form of tests.


In [ ]:
def calculate_tax_buggy(amount):
    return amount * 1.50

expected = 115
actual = calculate_tax_buggy(100)

try:
    assert actual == expected
except AssertionError:
    print(f"Logic test caught the problem: expected {expected}, got {actual}")


# 22. Mutation Testing with `mutmut`

Code coverage asks:

> Did the tests execute this line?

**Mutation testing** asks a stronger question:

> If we deliberately make a tiny mistake in this line, will the test suite notice?

A mutation tool might make small changes such as:

```text
>   →   >=
+   →   -
True → False
constant 10 → 11
```

If the tests fail, the mutant is **killed** — good.

If all tests still pass, the mutant **survives** — perhaps the test suite is weak.

Install and run:

```bash
python -m pip install mutmut
mutmut run
mutmut browse
```

### Windows note

Current `mutmut` documentation requires an operating system with `fork` support. On Windows, run it inside **WSL**.

Mutation testing is usually an advanced quality check rather than the first tool beginners use.


# 23. Automated Unit-Test Generation with Pynguin

**Pynguin** is an automated unit-test generation framework for Python.

It can inspect a Python module, execute it with generated inputs, and produce test suites.

Install:

```bash
python -m pip install pynguin
```

A conceptual command:

```bash
pynguin \
  --project-path ./my_project \
  --output-path ./generated_tests \
  --module-name math_utils
```

Pynguin can generate test suites in styles such as `pytest` or `unittest`.

### Very important safety warning

Pynguin **executes the module under test**, including imported code, with generated inputs.

Therefore:

- do not point it at unknown/untrusted code on your normal machine;
- inspect the code first;
- prefer an isolated environment/container for unknown code;
- its CLI requires the environment variable `PYNGUIN_DANGER_AWARE` to acknowledge this risk.

Automated generation is a useful assistant, but generated tests still need human review. A generated test can preserve current behaviour even when the current behaviour itself is wrong.


In [ ]:
pynguin_demo = Path("week09_testing_demo") / "triangle.py"
pynguin_demo.write_text(
"""def triangle(x: int, y: int, z: int) -> str:
    if x == y == z:
        return "equilateral"
    if x == y or x == z or y == z:
        return "isosceles"
    return "scalene"
""",
encoding="utf-8",
)

print("Prepared safe teaching module:", pynguin_demo)
print()
print("Example command to review before running:")
print(
    "pynguin --project-path week09_testing_demo "
    "--output-path generated_tests --module-name triangle"
)


# 24. A Practical Python Quality Toolbox

| Package/tool | Purpose | Install | Typical command |
|---|---|---|---|
| `unittest` | Unit-test framework | Built into Python | `python -m unittest` |
| `pytest` | Modern test runner/framework | `pip install pytest` | `pytest` |
| `pytest-cov` | Coverage with pytest | `pip install pytest-cov` | `pytest --cov=...` |
| `coverage` | Coverage engine/reporting | `pip install coverage` | `coverage run ...` |
| `hypothesis` | Property-based testing | `pip install hypothesis` | Used inside tests |
| `ruff` | Linting + formatting | `pip install ruff` | `ruff check .` / `ruff format .` |
| `mypy` | Static type checking | `pip install mypy` | `mypy .` |
| `bandit` | Security-pattern scanning | `pip install bandit` | `bandit -r .` |
| `radon` | Complexity/maintainability metrics | `pip install radon` | `radon cc -s .` |
| `mutmut` | Mutation testing | `pip install mutmut` | `mutmut run` |
| `pynguin` | Automated unit-test generation | `pip install pynguin` | `pynguin ...` |
| `pre-commit` | Run checks before commits | `pip install pre-commit` | `pre-commit run --all-files` |
| `black` | Dedicated formatter alternative | `pip install black` | `black .` |
| `pylint` | Additional static/lint analysis | `pip install pylint` | `pylint package_name` |
| `tox` / `nox` | Run tests in controlled environments | `pip install tox` / `nox` | project dependent |

You do **not** need every tool in every beginner project.

A sensible starter stack is:

```text
pytest
pytest-cov
ruff
```

Then add `mypy`, `hypothesis`, `bandit`, and `radon` as the project grows.


# 25. One Command Is Better Than Ten Manual Checks

A repeatable professional workflow might run:

```bash
ruff check .
ruff format --check .
mypy .
pytest --cov=. --cov-branch
bandit -r .
```

If these commands run automatically on every proposed change, developers receive feedback quickly.


# 26. Continuous Integration (CI)

**Continuous Integration** means that automated checks run when code changes — for example after a push or pull request.

```text
Developer changes code
        │
        ▼
Push / Pull Request
        │
        ▼
CI machine creates clean environment
        │
        ├──► install dependencies
        ├──► lint / format checks
        ├──► type checks
        ├──► unit tests
        ├──► coverage
        └──► security checks
        │
        ▼
PASS or FAIL report
```


## 26.1 Example GitHub Actions Workflow

```yaml
name: Python quality checks

on:
  push:
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - run: python -m pip install --upgrade pip

      - run: |
          pip install pytest pytest-cov ruff mypy bandit

      - run: ruff check .
      - run: ruff format --check .
      - run: mypy .
      - run: pytest --cov=. --cov-branch
      - run: bandit -r . -q
```

In a real project, dependencies are normally declared in a project/requirements file rather than copied into every CI command.


# 27. Test Design: What Should We Test?

For each unit, ask:

### 1. What is the normal behaviour?
The common case.

### 2. What are the boundaries?
Examples: exactly 0, exactly 18, exactly 50.

### 3. What unusual valid inputs exist?
Empty string? Negative number? Large number?

### 4. What invalid inputs should be rejected?
Wrong type? Impossible value?

### 5. What exceptions should be raised?
Use `assertRaises` or `pytest.raises`.

### 6. Does the unit have state?
Use fresh setup so tests remain independent.

### 7. What requirement would be costly if it broke?
Write a regression test for it.


# 28. Guided Practice 1 — `is_even`

Write tests for:

```python
def is_even(number):
    return number % 2 == 0
```

Test at least:

- `2`
- `3`
- `0`
- `-2`
- `-3`

First use plain `assert`. Then write a `unittest.TestCase`.


In [ ]:
# YOUR TURN

def is_even(number):
    return number % 2 == 0

# Plain assertions here:


### Suggested `unittest` solution


In [ ]:
class TestIsEven(unittest.TestCase):

    def test_positive_even(self):
        self.assertTrue(is_even(2))

    def test_positive_odd(self):
        self.assertFalse(is_even(3))

    def test_zero(self):
        self.assertTrue(is_even(0))

    def test_negative_even(self):
        self.assertTrue(is_even(-2))

    def test_negative_odd(self):
        self.assertFalse(is_even(-3))


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestIsEven)
unittest.TextTestRunner(verbosity=2).run(suite)


# 29. Guided Practice 2 — Circle Area

Requirements:

- `area = πr²`;
- radius must be non-negative;
- negative radius raises `ValueError`;
- radius `0` gives area `0`;
- floating-point comparisons should allow a small tolerance.

Use `unittest.assertAlmostEqual`.


In [ ]:
import math

def circle_area(radius):
    if radius < 0:
        raise ValueError("Radius cannot be negative")
    return math.pi * radius ** 2


class TestCircleArea(unittest.TestCase):

    def test_zero_radius(self):
        self.assertEqual(circle_area(0), 0)

    def test_radius_one(self):
        self.assertAlmostEqual(circle_area(1), math.pi)

    def test_radius_two(self):
        self.assertAlmostEqual(circle_area(2), 4 * math.pi)

    def test_negative_radius(self):
        with self.assertRaises(ValueError):
            circle_area(-1)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCircleArea)
unittest.TextTestRunner(verbosity=2).run(suite)


# 30. Guided Practice 3 — Password Rule

Requirement:

A password is accepted when:

- it is a string;
- it has at least 8 characters;
- it contains at least one digit.

Think about:

- `"abc"`
- `"abcdefgh"`
- `"abcdefg1"`
- `"12345678"`
- `""`
- `12345678`


In [ ]:
def is_valid_password(password):
    if not isinstance(password, str):
        return False
    return len(password) >= 8 and any(ch.isdigit() for ch in password)


assert is_valid_password("abcdefg1") is True
assert is_valid_password("12345678") is True
assert is_valid_password("abcdefgh") is False
assert is_valid_password("abc") is False
assert is_valid_password("") is False
assert is_valid_password(12345678) is False

print("Password tests passed.")


# 31. Challenge — Grade Calculator

Create tests for this specification:

| Mark | Grade |
|---:|---|
| 85–100 | HD |
| 75–84 | D |
| 65–74 | C |
| 50–64 | P |
| 0–49 | F |
| below 0 or above 100 | `ValueError` |

### Your mission

1. Write boundary tests **before** the implementation.
2. Include: `0, 49, 50, 64, 65, 74, 75, 84, 85, 100`.
3. Include `-1` and `101`.
4. Implement `calculate_grade`.
5. Run the suite.
6. Deliberately change one boundary and confirm the tests fail.


In [ ]:
# YOUR TURN

def calculate_grade(mark):
    # Replace with your implementation.
    raise NotImplementedError


# Write your tests below.


### Suggested solution


In [ ]:
def calculate_grade_solution(mark):
    if mark < 0 or mark > 100:
        raise ValueError("Mark must be between 0 and 100")
    if mark >= 85:
        return "HD"
    if mark >= 75:
        return "D"
    if mark >= 65:
        return "C"
    if mark >= 50:
        return "P"
    return "F"


class TestGradeCalculator(unittest.TestCase):

    def test_fail_boundaries(self):
        self.assertEqual(calculate_grade_solution(0), "F")
        self.assertEqual(calculate_grade_solution(49), "F")

    def test_pass_boundaries(self):
        self.assertEqual(calculate_grade_solution(50), "P")
        self.assertEqual(calculate_grade_solution(64), "P")

    def test_credit_boundaries(self):
        self.assertEqual(calculate_grade_solution(65), "C")
        self.assertEqual(calculate_grade_solution(74), "C")

    def test_distinction_boundaries(self):
        self.assertEqual(calculate_grade_solution(75), "D")
        self.assertEqual(calculate_grade_solution(84), "D")

    def test_hd_boundaries(self):
        self.assertEqual(calculate_grade_solution(85), "HD")
        self.assertEqual(calculate_grade_solution(100), "HD")

    def test_below_range(self):
        with self.assertRaises(ValueError):
            calculate_grade_solution(-1)

    def test_above_range(self):
        with self.assertRaises(ValueError):
            calculate_grade_solution(101)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestGradeCalculator)
unittest.TextTestRunner(verbosity=2).run(suite)


# 32. Common Unit-Testing Mistakes

1. **Only testing the happy path** — one normal case proves very little.
2. **Weak assertions** — checking only that a result is not `None` may miss wrong values.
3. **Tests depending on each other** — each test should normally run independently.
4. **Forgetting boundaries** — `49` and `50` may behave differently.
5. **Treating coverage as correctness** — high coverage is useful, but assertions still matter.
6. **Testing implementation details too aggressively** — prefer observable behaviour where possible.
7. **Non-deterministic tests** — uncontrolled time, random numbers, network access, or shared files can create flaky tests.
8. **Assuming a linter replaces tests** — static analysis and behavioural testing answer different questions.
9. **Automatically trusting generated tests** — generated tests need review against the actual requirement.


# 33. Testing Vocabulary Cheat Sheet

| Term | Meaning |
|---|---|
| **Test case** | One defined check with input/conditions and expected behaviour |
| **Test suite** | Collection of test cases |
| **Test runner** | Tool that discovers/executes tests and reports results |
| **Unit** | Small independently testable part of a program |
| **Assertion** | Statement comparing expected behaviour with actual behaviour |
| **Fixture** | Reusable setup/state for tests |
| **Mock** | Controlled replacement for a dependency |
| **Regression** | A previously working behaviour becomes broken |
| **Coverage** | Measure of code executed by tests |
| **Branch coverage** | Whether alternative control-flow branches were exercised |
| **Property-based test** | Tests general properties using generated inputs |
| **Mutation testing** | Deliberately changes code to measure test-suite sensitivity |
| **Static analysis** | Inspects source without normal execution |
| **Linting** | Detects suspicious/style-related source patterns |
| **Formatting** | Rewrites layout/style consistently |
| **Type checking** | Checks annotated type consistency |
| **CI** | Automatically runs checks when code changes |


# 34. Recommended Workflow for a Small Python Project

```text
1. Read the requirement
       │
       ▼
2. Identify one small unit
       │
       ▼
3. Write normal + boundary + invalid test cases
       │
       ▼
4. Implement / modify code
       │
       ▼
5. Run pytest
       │
       ├── FAIL → inspect → fix → rerun
       │
       ▼
6. Check coverage
       │
       ▼
7. Run Ruff / mypy / Bandit as appropriate
       │
       ▼
8. Commit code
       │
       ▼
9. CI repeats checks automatically
```

The goal is **fast feedback**.


# 35. Final Self-Check Quiz

The quiz covers:

- manual vs automated testing;
- assertions;
- units and test cases;
- `unittest`;
- `pytest`;
- parametrization and fixtures;
- Hypothesis;
- coverage;
- Ruff;
- mypy;
- Bandit;
- Radon;
- mutation testing;
- Pynguin;
- CI.

Try to answer without scrolling back.


In [ ]:
quiz_questions = [
    {
        "question": "1. What is the main purpose of automated testing?",
        "options": [
            "To make code colourful",
            "To repeatedly verify behaviour against expectations",
            "To replace all manual testing",
            "To remove every exception",
        ],
        "answer": 1,
        "explanation": "Automated tests repeatedly compare actual behaviour with expected behaviour."
    },
    {
        "question": "2. If a normal Python assert condition is False, what happens?",
        "options": [
            "Python returns False and continues",
            "A SyntaxError is raised",
            "An AssertionError is raised",
            "The function is deleted",
        ],
        "answer": 2,
        "explanation": "A failed assert raises AssertionError."
    },
    {
        "question": "3. Which best describes a unit in unit testing?",
        "options": [
            "Always the whole application",
            "A small independently testable piece such as a function",
            "Only a database",
            "Only a user interface",
        ],
        "answer": 1,
        "explanation": "A unit is usually a small function, method, or class that can be tested in isolation."
    },
    {
        "question": "4. In Arrange–Act–Assert, what is the Act step?",
        "options": [
            "Prepare inputs",
            "Call the code being tested",
            "Compare actual and expected values",
            "Install pytest",
        ],
        "answer": 1,
        "explanation": "Act means executing the behaviour under test."
    },
    {
        "question": "5. Which class is normally inherited when writing unittest tests?",
        "options": [
            "unittest.TestCase",
            "unittest.Unit",
            "pytest.TestCase",
            "TestRunner.Base",
        ],
        "answer": 0,
        "explanation": "unittest tests commonly inherit from unittest.TestCase."
    },
    {
        "question": "6. Which unittest assertion checks that two values are equal?",
        "options": ["assertSame", "assertValue", "assertEqual", "assertOutput"],
        "answer": 2,
        "explanation": "self.assertEqual(actual, expected) checks equality."
    },
    {
        "question": "7. How do we normally test an expected exception in unittest?",
        "options": [
            "with self.assertRaises(SomeError):",
            "with self.expectCrash():",
            "assert exception",
            "except unittest:",
        ],
        "answer": 0,
        "explanation": "assertRaises is designed for expected exceptions."
    },
    {
        "question": "8. Why is pytest.mark.parametrize useful?",
        "options": [
            "It installs packages",
            "It runs one test function with multiple sets of data",
            "It formats Python source",
            "It measures security vulnerabilities",
        ],
        "answer": 1,
        "explanation": "Parametrization expresses many cases as data without duplicating the test body."
    },
    {
        "question": "9. What is a pytest fixture primarily used for?",
        "options": [
            "Reusable test setup/data",
            "Converting Python to Java",
            "Changing font size",
            "Deleting failed tests",
        ],
        "answer": 0,
        "explanation": "Fixtures provide reusable setup, objects, files, or other test resources."
    },
    {
        "question": "10. What does Hypothesis add to testing?",
        "options": [
            "Automatic formatting",
            "Property-based testing with generated inputs",
            "Only syntax checking",
            "Git commits",
        ],
        "answer": 1,
        "explanation": "Hypothesis generates inputs from strategies to test general properties."
    },
    {
        "question": "11. What does code coverage tell us?",
        "options": [
            "That the program is definitely correct",
            "Which code was exercised while tests ran",
            "Who wrote the code",
            "Whether comments are grammatically correct",
        ],
        "answer": 1,
        "explanation": "Coverage measures execution, not correctness."
    },
    {
        "question": "12. Why does 100% coverage NOT guarantee correctness?",
        "options": [
            "Coverage cannot run Python",
            "Tests may execute every line but use weak or wrong assertions",
            "Coverage only works on Java",
            "100% is mathematically impossible",
        ],
        "answer": 1,
        "explanation": "Execution alone does not prove that the expected behaviour was checked correctly."
    },
    {
        "question": "13. What is Ruff mainly used for in this notebook?",
        "options": ["Linting and formatting", "Database backup", "Unit-test generation", "Neural-network training"],
        "answer": 0,
        "explanation": "Ruff provides fast Python linting and formatting."
    },
    {
        "question": "14. What does mypy primarily check?",
        "options": ["Image resolution", "Static type consistency", "Network speed", "Code coverage"],
        "answer": 1,
        "explanation": "mypy statically checks annotated Python types."
    },
    {
        "question": "15. What does Bandit focus on?",
        "options": [
            "Common security-risk patterns in Python",
            "Formatting markdown",
            "Generating random passwords for users",
            "Measuring line coverage",
        ],
        "answer": 0,
        "explanation": "Bandit performs security-oriented static analysis."
    },
    {
        "question": "16. What can Radon measure?",
        "options": [
            "Cyclomatic complexity and maintainability metrics",
            "Only CPU temperature",
            "GitHub stars",
            "Unit-test assertions",
        ],
        "answer": 0,
        "explanation": "Radon measures code metrics including cyclomatic complexity and maintainability."
    },
    {
        "question": "17. What is mutation testing asking?",
        "options": [
            "Can we rename every variable?",
            "Would the test suite detect small intentional code changes?",
            "Can Python compile to C?",
            "Does the code use enough comments?",
        ],
        "answer": 1,
        "explanation": "Mutation testing changes code slightly and checks whether tests fail."
    },
    {
        "question": "18. What is Pynguin?",
        "options": [
            "A formatter",
            "An automated unit-test generation framework for Python",
            "A database",
            "A package manager",
        ],
        "answer": 1,
        "explanation": "Pynguin generates unit tests for Python modules."
    },
    {
        "question": "19. Why must Pynguin be used carefully?",
        "options": [
            "It cannot read Python",
            "It executes the module under test with generated inputs",
            "It requires a web browser",
            "It always deletes tests",
        ],
        "answer": 1,
        "explanation": "Pynguin executes target/imported code, so unknown code should be isolated and reviewed."
    },
    {
        "question": "20. Which is most likely to catch a validly typed discount formula that adds 20% instead of subtracting it?",
        "options": [
            "A behavioural unit test with the correct expected result",
            "A formatter alone",
            "A type checker alone",
            "A syntax parser alone",
        ],
        "answer": 0,
        "explanation": "Valid syntax and types do not guarantee correct business logic."
    },
    {
        "question": "21. What is a boundary value?",
        "options": [
            "A value near a point where behaviour changes, such as 49/50",
            "A random file name",
            "A Python keyword",
            "A testing framework",
        ],
        "answer": 0,
        "explanation": "Boundaries are transition points where off-by-one errors commonly occur."
    },
    {
        "question": "22. Why should tests usually be independent?",
        "options": [
            "So their result does not depend on another test running first",
            "So they use more memory",
            "So they cannot be automated",
            "So no assertions are needed",
        ],
        "answer": 0,
        "explanation": "Independent tests are repeatable and can run in any order."
    },
    {
        "question": "23. What is Continuous Integration (CI) used for here?",
        "options": [
            "Automatically running quality checks when code changes",
            "Replacing Python with another language",
            "Writing lecture notes manually",
            "Hiding test failures",
        ],
        "answer": 0,
        "explanation": "CI repeatedly runs automated checks on pushes/pull requests or other events."
    },
    {
        "question": "24. Which is the best statement about linters and tests?",
        "options": [
            "A linter completely replaces tests",
            "Tests completely replace all static analysis",
            "They answer different questions and are useful together",
            "Neither is useful",
        ],
        "answer": 2,
        "explanation": "Static analysis and behavioural testing complement each other."
    },
]

def launch_widget_quiz():
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError:
        print("ipywidgets is not available.")
        print("Use the text-based fallback quiz in the next cell.")
        return

    state = {"index": 0, "score": 0, "checked": set()}

    title = widgets.HTML()
    progress = widgets.IntProgress(
        value=1, min=1, max=len(quiz_questions), description="Question"
    )
    radio = widgets.RadioButtons(
        options=[], description="", layout=widgets.Layout(width="95%")
    )
    feedback = widgets.HTML()
    check_button = widgets.Button(description="Check Answer", button_style="primary")
    next_button = widgets.Button(description="Next Question")
    reset_button = widgets.Button(description="Reset Quiz")
    score_box = widgets.HTML()

    def render():
        q = quiz_questions[state["index"]]
        title.value = f"<h3>{q['question']}</h3>"
        radio.options = q["options"]
        radio.value = None
        feedback.value = ""
        progress.value = state["index"] + 1
        score_box.value = (
            f"<b>Score:</b> {state['score']} / {len(state['checked'])} checked &nbsp; "
            f"<b>Progress:</b> {state['index'] + 1}/{len(quiz_questions)}"
        )

    def check_answer(_):
        idx = state["index"]
        if radio.value is None:
            feedback.value = "Please select an answer first."
            return

        q = quiz_questions[idx]
        selected_index = list(q["options"]).index(radio.value)

        if idx not in state["checked"]:
            state["checked"].add(idx)
            if selected_index == q["answer"]:
                state["score"] += 1

        if selected_index == q["answer"]:
            feedback.value = f"<b>Correct.</b> {q['explanation']}"
        else:
            correct = q["options"][q["answer"]]
            feedback.value = (
                f"<b>Not quite.</b> Correct answer: <b>{correct}</b><br>"
                f"{q['explanation']}"
            )

        score_box.value = (
            f"<b>Score:</b> {state['score']} / {len(state['checked'])} checked &nbsp; "
            f"<b>Progress:</b> {state['index'] + 1}/{len(quiz_questions)}"
        )

    def next_question(_):
        if state["index"] < len(quiz_questions) - 1:
            state["index"] += 1
            render()
        else:
            feedback.value = (
                f"<h3>Quiz complete</h3>"
                f"Checked questions: {len(state['checked'])}/{len(quiz_questions)}<br>"
                f"Final score on checked questions: "
                f"<b>{state['score']}/{len(state['checked'])}</b>"
            )

    def reset_quiz(_):
        state["index"] = 0
        state["score"] = 0
        state["checked"] = set()
        render()

    check_button.on_click(check_answer)
    next_button.on_click(next_question)
    reset_button.on_click(reset_quiz)

    render()
    display(
        widgets.VBox([
            progress,
            title,
            radio,
            widgets.HBox([check_button, next_button, reset_button]),
            feedback,
            score_box,
        ])
    )

launch_widget_quiz()


## Text-Based Quiz Fallback

If widgets do not render, run the next cell.

Enter the option number shown on screen. Type `q` to stop.


In [ ]:
def run_text_quiz():
    score = 0
    answered = 0

    for q in quiz_questions:
        print("\n" + "=" * 80)
        print(q["question"])
        for i, option in enumerate(q["options"], start=1):
            print(f"{i}. {option}")

        while True:
            raw = input("Your answer (number, or q to quit): ").strip().lower()
            if raw == "q":
                print(f"\nStopped. Score: {score}/{answered}")
                return
            if raw.isdigit() and 1 <= int(raw) <= len(q["options"]):
                break
            print("Please enter a valid option number.")

        answered += 1
        chosen = int(raw) - 1

        if chosen == q["answer"]:
            score += 1
            print("Correct.")
        else:
            print("Incorrect.")
            print("Correct answer:", q["options"][q["answer"]])

        print(q["explanation"])

    print("\n" + "=" * 80)
    print(f"Final score: {score}/{answered}")

# Uncomment to start:
# run_text_quiz()


# 36. Final Summary

This week moved from checking code manually to building repeatable automated evidence.

```text
Manual checking
      ↓
assert
      ↓
unittest
      ↓
pytest
      ↓
parametrized + property-based tests
      ↓
coverage
      ↓
linting / formatting / typing / security / complexity
      ↓
mutation testing and automated test generation
      ↓
Continuous Integration
```

Remember:

> **Tools can tell us many things about source code, but correctness ultimately depends on the requirement. Good tests encode that requirement as executable checks.**

For the next practical session, take one small function from an earlier week and build:

1. at least five test cases;
2. one boundary case;
3. one invalid/exception case;
4. a `unittest` suite;
5. a `pytest` version;
6. a coverage report;
7. a Ruff scan.


# 37. Further Reading — Official Documentation

- Python `unittest`: https://docs.python.org/3/library/unittest.html
- pytest: https://docs.pytest.org/
- Hypothesis: https://hypothesis.readthedocs.io/
- coverage.py: https://coverage.readthedocs.io/
- Ruff: https://docs.astral.sh/ruff/
- mypy: https://mypy.readthedocs.io/
- Bandit: https://bandit.readthedocs.io/
- Radon: https://radon.readthedocs.io/
- mutmut: https://mutmut.readthedocs.io/
- Pynguin: https://pynguin.readthedocs.io/
